Here is the code where you put in H, M, G, model and measurement

In [66]:
#| default_exp prediction_module

In [67]:
#| export
import pickle
import pandas as pd 
from anthropmass.data_module import *
from anthropmass.anthro_module import *
from anthropmass.bambi_module import *

This function is normalizing the persons weight and height

In [68]:
#| export
def Delete_normalize_person(weight, height, gender):
    person=pd.DataFrame({'weightkg': [weight], 'stature': [height], 'Gender': [gender]})
    normalize(person, 'weightkg')
    normalize(person, 'stature')
    return person

DELETETThis functions substract the ansur mean of weight and stature.

In [69]:
#| export
def Delete_minus_mean_person(weight, height, gender):
    person=pd.DataFrame({'weightkg': [weight], 'stature': [height], 'Gender': [gender]})
    minus_mean(person,['weightkg','stature'])
    return person

This functions gets the pickled model

In [70]:
#| export
def get_pickled_model(kindofmodel:str, measurement:str):
    filepath = f'../output/anthro_models/{kindofmodel}/pickle_{measurement}_{kindofmodel}'
    with open(filepath,'rb') as file:
        model=pickle.load(file)
    return model

Predict_from_model uses the pickled model and the "minus mean person" to predict the measurement. One of the arguments passed into this functions is "kind of model", and currently there are three different models to chose between. The three models are xgboost, bambi, and bambi with component as input. When using bambi with component as input, the component needs to be passed as an argument into the functions, otherwise Army Reserve is set as default. The three different components to pass into the function are Regular Army, Army National Guard, and Army Reserve.

In [71]:
#| export
def predict_from_model(pickledmodel, kindofmodel:str, measurement:str, input_person:dict, c=False):
    person = minus_mean(input_person,['weightkg','stature'])
    if kindofmodel=='xgboost':
        return pickledmodel.predict(person)
    
    elif kindofmodel=='bambi':
        formula = 'weightkg + stature + C(Gender)'
        model = make_model_formula(measurement, formula)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    
    elif kindofmodel=='bambi_c':
        if isinstance(c, pd.Series):
            person['Component']=c
        elif c!= False:
            person['Component']=c
        
        else:
            person['Component']='Army Reserve'
        formula='0 + C(Gender) + Component + weightkg + stature'
        model = make_model_formula(measurement, formula)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    
    else:
        return 'wrong model name'

In the following function the function above is called upon and looped through for all passed measurements.

In [72]:
#| export
def DELETE_make_predictions(kindofmodel:str, measurements:list, w, h, g, c=False):
    output=pd.DataFrame()
    for m in measurements:
        pred = predict_from_model(kindofmodel, m, w, h, g, c=False)
        add_to_df(output, m, pred)
    return output

Predict for group is a function that itterates through a dataframe with multiple people. THe argument n is used when you only want to predict a part of the dataframe. 

In [73]:
#| export
def DELETE_predict_for_group(kindofmodel:str, measurements:list, group:dict, n=False):
    preds_all=pd.DataFrame()
    for index, row in group.iterrows():
        if index%10 == 0:
            print('index:', index)
        if kindofmodel == 'bambi_c':
            pred_row = make_predictions(kindofmodel, measurements, row['weightkg'], row['stature'], row['Gender'], row['Component'])
        else:
            pred_row = make_predictions(kindofmodel, measurements, row['weightkg'], row['stature'], row['Gender'])
        preds_all = pd.concat([preds_all, pred_row], ignore_index=True)
        if index == n:
            return preds_all
    return preds_all

In [74]:
def predict_column(kindofmodel:str, measurement:str, group:dict, c=False):
    pickledmodel = get_pickled_model(kindofmodel, measurement)
    preds = predict_from_model(
    pickledmodel,
    kindofmodel,
    measurement,
    group[['weightkg','stature','Gender']], c)

    return preds

In [75]:
def loop_measurements(kindofmodel:str, measurements:list, group:dict, c=False):
    preds_all=pd.DataFrame()
    for m in measurements:
        preds_all[m]=predict_column(kindofmodel, m, group, c)
    return preds_all

om inte comp då välja skriva str valfri c, men om har comp lägg in df

In [76]:
test= pd.read_csv('../data/processed/ANSURIItest.csv')
all_names= measurement_names()

In [77]:
xgboost=loop_measurements('xgboost', all_names, test)
test.to_csv('../output/anthro_results/predict_all_test_xgboost.csv', index=False)

C:\Users\sanna\Desktop\kandidat\bsp_estimation\anthropmass\anthropmass\data_module.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[measurement] = data[measurement] - avg_m
C:\Users\sanna\Desktop\kandidat\bsp_estimation\anthropmass\anthropmass\data_module.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[measurement] = data[measurement] - avg_m
C:\Users\sanna\Desktop\kandidat\bsp_estimation\anthropmass\anthropmass\data_module.py:22: SettingWithCopyWarning: 
A value is trying to be set on a 

In [78]:
bambi=loop_measurements('bambi', all_names, test)
test.to_csv('../output/anthro_results/predict_all_test_bambi.csv', index=False)

C:\Users\sanna\Desktop\kandidat\bsp_estimation\anthropmass\anthropmass\data_module.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[measurement] = data[measurement] - avg_m
C:\Users\sanna\Desktop\kandidat\bsp_estimation\anthropmass\anthropmass\data_module.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[measurement] = data[measurement] - avg_m
C:\Users\sanna\Desktop\kandidat\bsp_estimation\anthropmass\anthropmass\data_module.py:22: SettingWithCopyWarning: 
A value is trying to be set on a 

KeyboardInterrupt: 

In [ ]:
bambi_c=loop_measurements('bambi_c', all_names, test, test['Component'])
test.to_csv('../output/anthro_results/predict_all_test_bambi_c.csv', index=False)

In [ ]:
bambi_c_AR=loop_measurements('bambi_c', all_names, test, 'Army Reserved')
test.to_csv('../output/anthro_results/predict_all_test_bambi_c_AR.csv', index=False)